# Monte Carlo check of the digital cluster model

Track footprint on the readout plane: segment [a, a+L], uniform charge density.
Pixel k = [k p, (k+1) p]. Pixel fires if its charge > r * Qtot (r = threshold fraction).
Optional Gaussian diffusion sigma_d applied to the charge profile.

Note: [`2026-10-20_cluster-error-model.md`](2026-10-20_cluster-error-model.md).

In [ ]:
import numpy as np
from math import erf, sqrt

rng = np.random.default_rng(1)

## Pixel charges

Charge fraction per pixel for a box footprint, optionally convolved with a Gaussian.

In [ ]:
def pixel_charges(a, L, p, sd, kmin, kmax):
    # charge fraction in pixels kmin..kmax for a box [a,a+L] convolved with N(0,sd)
    ks = np.arange(kmin, kmax + 1)
    lo = ks * p
    hi = (ks + 1) * p
    if sd <= 0:
        ov = np.clip(np.minimum(hi, a + L) - np.maximum(lo, a), 0, None)
        return ov / L
    # integral over pixel of [Phi((x-a)/sd) - Phi((x-a-L)/sd)] / L
    def F(x):  # antiderivative of Phi(x/sd): x*Phi + sd*phi
        z = x / sd
        Phi = 0.5 * (1 + np.vectorize(erf)(z / sqrt(2)))
        phi = np.exp(-0.5 * z * z) / sqrt(2 * np.pi)
        return x * Phi + sd * phi
    val = (F(hi - a) - F(lo - a)) - (F(hi - a - L) - F(lo - a - L))
    return val / L

## Cluster size and residual

MC over the footprint midpoint, compared with the analytic P(N) and σ_N.

In [ ]:
def run(p=50.0, t=150.0, alpha_deg=10.0, r=0.08, sd=0.0, n=200000):
    L = t * np.tan(np.radians(alpha_deg))
    xm = rng.uniform(0, p, n)           # footprint midpoint, uniform over one pitch
    a = xm - L / 2
    kmin = int(np.floor((a.min() - 5 * sd) / p)) - 1
    kmax = int(np.ceil((a.max() + L + 5 * sd) / p)) + 1
    ks = np.arange(kmin, kmax + 1)
    res = {}
    for i in range(n):
        q = pixel_charges(a[i], L, p, sd, kmin, kmax)
        fired = ks[q > r]
        if fired.size == 0:
            continue
        N = fired.max() - fired.min() + 1
        centre = (fired.min() + fired.max() + 1) * p / 2
        res.setdefault(N, []).append(xm[i] - centre)
    lam = L / p
    lam_eff = lam * (1 - 2 * r)
    nn = int(np.floor(lam_eff)); ph = lam_eff - nn
    print(f"p={p} t={t} alpha={alpha_deg} r={r} sd={sd}  L={L:.1f}um lam={lam:.3f} lam_eff={lam_eff:.3f} (n={nn}, phi={ph:.3f})")
    print(f"  analytic: N={nn+1}: P={1-ph:.3f} sigma={(1-ph)*p/sqrt(12):.2f}   N={nn+2}: P={ph:.3f} sigma={ph*p/sqrt(12):.2f}   p/sqrt12={p/sqrt(12):.2f}")
    for N in sorted(res):
        d = np.array(res[N])
        print(f"  MC: N={N}: P={d.size/n:.3f} mean={d.mean():+.2f} rms={d.std():.2f} width(max-min)={d.max()-d.min():.1f}")
    return res

## Pixel, no diffusion

In [ ]:
run(alpha_deg=0.01, r=0.0)

In [ ]:
run(alpha_deg=10, r=0.0)

In [ ]:
run(alpha_deg=10, r=0.08)

In [ ]:
run(alpha_deg=20, r=0.08)

## Pixel, diffusion σ_d = 5 µm

In [ ]:
run(alpha_deg=10, r=0.08, sd=5.0, n=40000)

In [ ]:
run(alpha_deg=2, r=0.08, sd=5.0, n=40000)

## SCT like

In [ ]:
run(p=80, t=285, alpha_deg=10, r=0.29, sd=0.0)

In [ ]:
run(p=80, t=285, alpha_deg=24, r=0.29, sd=0.0)